In [2]:
import numpy as np
from prettytable import PrettyTable

# Define users
users = ["Rahul", "Priya", "Arun", "Divya"]
expense_matrix = np.zeros((len(users), len(users)))  # Rows: Payer, Columns: Beneficiary
transaction_history = []

def add_expense(payer, beneficiaries, amount, description="General Expense", split_type="equal", custom_shares=None):
    payer_idx = users.index(payer)

    if split_type == "equal":
        share_per_person = amount / len(beneficiaries)
        shares = {b: share_per_person for b in beneficiaries}
    elif split_type == "custom" and custom_shares:
        if sum(custom_shares) != amount:
            raise ValueError("Custom shares must sum up to the total amount!")
        shares = {beneficiaries[i]: custom_shares[i] for i in range(len(beneficiaries))}
    else:
        raise ValueError("Invalid split type or missing custom shares.")

    for beneficiary, share in shares.items():
        if beneficiary != payer:
            b_idx = users.index(beneficiary)
            expense_matrix[payer_idx][b_idx] += share

    # Maintain transaction history for each user
    transaction_history.append({
        "payer": payer,
        "amount": amount,
        "description": description,
        "beneficiaries": beneficiaries
    })

def calculate_settlements():
    total_paid = np.sum(expense_matrix, axis=1)  # How much each person paid for others
    total_owed = np.sum(expense_matrix, axis=0)  # How much others paid for them
    net_balance = total_paid - total_owed        # Positive: should receive, Negative: owes
    return net_balance

def display_settlements():
    balances = calculate_settlements()
    table = PrettyTable()
    table.field_names = ["User", "Net Balance Status"]

    for i, user in enumerate(users):
        if balances[i] > 0.01:
            table.add_row([user, f"Should Receive ₹{balances[i]:.2f}"])
        elif balances[i] < -0.01:
            table.add_row([user, f"Owes ₹{-balances[i]:.2f}"])
        else:
            table.add_row([user, "Is Settled"])

    print("\n--- Final Settlements Summary ---")
    print(table)

def display_history():
    print("\n--- Transaction History ---")
    for idx, txn in enumerate(transaction_history, 1):
        print(f"{idx}. {txn['payer']} paid ₹{txn['amount']} for '{txn['description']}' (Shared among: {', '.join(txn['beneficiaries'])})")

def suggest_payments():
    balances = calculate_settlements()
    creditors = [(users[i], amt) for i, amt in enumerate(balances) if amt > 0.01]
    debtors = [(users[i], -amt) for i, amt in enumerate(balances) if amt < -0.01]

    print("\n--- Suggested Transactions to Clear Debts ---")
    settlements_made = False

    while debtors and creditors:
        debtor, debt_amt = debtors.pop(0)
        creditor, credit_amt = creditors.pop(0)

        settle_amount = min(debt_amt, credit_amt)
        print(f"{debtor} should pay ₹{settle_amount:.2f} to {creditor}")
        settlements_made = True

        debt_amt -= settle_amount
        credit_amt -= settle_amount

        if debt_amt > 0.01:
            debtors.insert(0, (debtor, debt_amt))
        if credit_amt > 0.01:
            creditors.insert(0, (creditor, credit_amt))

    if not settlements_made:
        print("No transactions needed. Everyone is settled.")


In [3]:
# 1. Rahul pays for dinner (Equal split among all 4 friends)
add_expense("Rahul", ["Rahul", "Priya", "Arun", "Divya"], 1600, description="Team Dinner", split_type="equal")

# 2. Priya pays for cab booking (Custom split among Rahul, Priya, and Arun)
add_expense("Priya", ["Rahul", "Priya", "Arun"], 900, description="Cab Booking", split_type="custom", custom_shares=[300, 400, 200])

# 3. Arun pays for snacks (Equal split between Arun and Divya)
add_expense("Arun", ["Arun", "Divya"], 500, description="Evening Snacks", split_type="equal")

# Display results
display_history()
display_settlements()
suggest_payments()



--- Transaction History ---
1. Rahul paid ₹1600 for 'Team Dinner' (Shared among: Rahul, Priya, Arun, Divya)
2. Priya paid ₹900 for 'Cab Booking' (Shared among: Rahul, Priya, Arun)
3. Arun paid ₹500 for 'Evening Snacks' (Shared among: Arun, Divya)

--- Final Settlements Summary ---
+-------+------------------------+
|  User |   Net Balance Status   |
+-------+------------------------+
| Rahul | Should Receive ₹900.00 |
| Priya | Should Receive ₹100.00 |
|  Arun |      Owes ₹350.00      |
| Divya |      Owes ₹650.00      |
+-------+------------------------+

--- Suggested Transactions to Clear Debts ---
Arun should pay ₹350.00 to Rahul
Divya should pay ₹550.00 to Rahul
Divya should pay ₹100.00 to Priya
